### Chemical class

A `chemical` class has been defined for individual chemicals. This class has methods that determine the vapor pressure and boiling point of the pure chemical.

Verify that the given vapor pressure coefficients for methyl formate and methyl acetate are consistent with the normal boiling points of the pure chemicals, 32.0°C and 56.9°C, respectively.

In [1]:
# import modules
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

# internal units
# pressure: Pa
# temperature: K

# unit conversions
atm_to_Pa = 101325
bar_to_Pa = 1.e5
Pa_to_bar = 1.e-5
degC_to_K = 273.15

# define classes
class chemical(object):

  def __init__(self,name,formula,coeffs):
    self.name = name
    self.formula = formula
    self.coeffs = coeffs

  def Psat(self,T):
    A,B,C,D,E = self.coeffs
    return np.exp(A+B/T+C*np.log(T)+D*T**E)

  def P_res(self,T,P):
    res = self.Psat(T) - P
    return res

  def Tsat(self,P=101325,T0=298):
    return opt.root(self.P_res,T0,args=(P)).x[0]
  
  def __str__(self):
    return  ('Name: ' + self.name + '\n' +
             'Formula: ' + self.formula)

# main code
mf = chemical('methyl formate','C2H4O2',
              [71.57,-5430,-7.4848,6.22e-6,2])
ma = chemical('methyl acetate','C3H6O2',
              [61.267,-5618.6,-5.6473,2.108e-17,6])

In [2]:
print(f'Boiling Point for methyl formate: {chemical.Tsat(mf)-degC_to_K:.1f}', "C") 
print(f'Boiling Point for methyl acetate: {chemical.Tsat(ma)-degC_to_K:.1f}', "C") 

Boiling Point for methyl formate: 31.7 C
Boiling Point for methyl acetate: 57.1 C


### Mixture class

Create a `mixture` class with the following attributes:

- `species` - list of `chemical` instances
- `mols` - numpy array, amounts of the elements in `species`
- `mol_frac` - numpy array, mole fractions of the elements in `species`

Add the following functions to the `chemical` class:

- `__rmul__(self,other)` - where `self` is a `chemical` object and `other` is an integer or float. This function creates a `mixture` object and sets `species = [self]`, `mols = np.array([other])`, and `mol_frac = np.array([1.0])`.

Add the following functions to the `cixture` class:

- `__add__(self,other)` - where `self` and `other` are both `mixture` objects. This function creates a new `mixture` object, concatenates `self.species` and `other.species`, and joins `self.mols` and `other.mols` using `np.hstack()`. It also calculates the mole fraction.
- `dew_point(self,P)` - evaluates the dew point temperature of the mixture at the specified pressure from the Rachford-Rice equation  
$ \sum_i z_i \left(1 - \frac{1}{K_i}\right) = 0 $  
where $z_i$ are the mole fractions of the mixture, and $K_i = P_{sat}(T)/P$.
- `bbl_point(self,P)` - evaluates the bubble point temperature of the mixture at the specified pressure from the Rachford-Rice equation
$$ \sum_i z_i \left(K_i - 1\right) = 0 $$

Note that there is no `__init__()` function. New `mixture` instances are only created through other functions.

Create the following mixtures and evaluate the normal dew point and bubble point temperatures of each:

```
# Mixtures
equimolar = 1*mf + 1*ma
one_nine = 0.1*mf + 0.9*ma
```

In [3]:
# import modules
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

# internal units
# pressure: Pa
# temperature: K

# unit conversions
atm_to_Pa = 101325
bar_to_Pa = 1.e5
Pa_to_bar = 1.e-5
degC_to_K = 273.15

# define classes
class chemical(object):

  def __init__(self,name,formula,coeffs):
    self.name = name
    self.formula = formula
    self.coeffs = coeffs

  def Psat(self,T):
    A,B,C,D,E = self.coeffs
    return np.exp(A+B/T+C*np.log(T)+D*T**E)

  def P_res(self,T,P):
    res = self.Psat(T) - P
    return res

  def Tsat(self,P=101325,T0=298):
    return opt.root(self.P_res,T0,args=(P)).x[0]
  
  def __str__(self):
    return  ('Name: ' + self.name + '\n' +
             'Formula: ' + self.formula)
  
  def __rmul__(self,other):
     mix_obj = mixture()
     mix_obj.species = [self]
     mix_obj.mols = np.array([other])
     mix_obj.mol_frac = np.array([1.0])
     return mix_obj
  
class mixture(object):
    def __add__(self,other):
        z = mixture()
        z.species = self.species + other.species
        z.mols = np.hstack((self.mols,other.mols))
        total_mols = np.sum(z.mols)
        z.mol_frac = z.mols/total_mols
        return z
    
    def dew_point(self,P):
        z = self.mol_frac
        f = self.species
        def func(T):
            Psat = np.array([float(i.Psat(T)) for i in f])
            rr = np.sum(z*(1-(1/(Psat/P))))
            return rr
        initial_guess = 300
        Tdew = opt.fsolve(func,initial_guess)
        return Tdew[0]

    def bbl_point(self,P):
       z = self.mol_frac
       f = self.species
       def func(T):
            Psat = np.array([float(i.Psat(T)) for i in f])
            return np.sum(z*((Psat/P)-1))
       initial_guess = 300
       Tbbl = opt.fsolve(func,initial_guess)
       return Tbbl[0]
    
# Main Code 

mf = chemical('methyl formate','C2H4O2',
              [71.57,-5430,-7.4848,6.22e-6,2])
ma = chemical('methyl acetate','C3H6O2',
              [61.267,-5618.6,-5.6473,2.108e-17,6])

# Mixture
#A = chemical.__rmul__(ma,1)
#B = chemical.__rmul__(mf,1)
#mix1 = mixture.__add__(A,B)

#A1 = chemical.__rmul__(ma,0.1)
#B2 = chemical.__rmul__(mf,0.9)
#mix2 = mixture.__add__(A1,B2)

mix1 = 1*mf + 1*ma  # Equimolar mixture
mix2 = 0.1*mf + 0.9*ma  # One to Nine

print(f'Dew Point for Equimolar Mixture: {mix1.dew_point(101325)-degC_to_K:.3f}', "C") 
print(f'Bubble Point for Equimolar Mixture:  {mix1.bbl_point(101325)-degC_to_K:.3f}', "C") 
print(f'Dew Point for One to Nine Mixture: {mix2.dew_point(101325)-degC_to_K:.3f}', "C") 
print(f'Bubble Point for One to Nine Mixture:  {mix2.bbl_point(101325)-degC_to_K:.3f}', "C") 


Dew Point for Equimolar Mixture: 47.286 C
Bubble Point for Equimolar Mixture:  41.680 C
Dew Point for One to Nine Mixture: 55.345 C
Bubble Point for One to Nine Mixture:  53.314 C


C:\Users\selvi\AppData\Local\Temp\ipykernel_9392\309774124.py:59: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Psat = np.array([float(i.Psat(T)) for i in f])
C:\Users\selvi\AppData\Local\Temp\ipykernel_9392\309774124.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Psat = np.array([float(i.Psat(T)) for i in f])
